# Evaluation der Korpusanreicherung

Wie gut können die Modelle, verglichen mit einem 20 Rezensionen umfassenden Ground Truth Datensatz, Metadaten annotieren und Sentiment klassifizieren?

In [ ]:
#loading the dataframe
import pandas as pd
df = pd.read_csv("reviews_with_ground_truth - review_annotation_test (1).csv")

### Gewünschtes Format der Daten

Die Daten der Extraktions- bzw. Klassifikationsaufgabe sollen, damit sie geparst und ausgewertet werden können, in xml-Tags gesetzt werden wie folgt:

```
<review>

<review_metadata>

<sentiment>
  <sentiment_overall>positive|negative|ambivalent|neutral</sentiment_overall>
  <sentiment_style>positive|negative|ambivalent|neutral|not_mentioned</sentiment_style>
  <sentiment_content>positive|negative|ambivalent|neutral|not_mentioned</sentiment_content>
<sentiment_society>positive|negative|ambivalent|neutral|not_mentioned</sentiment_society>
 <sentiment_morality>positive|negative|ambivalent|neutral|not_mentioned</sentiment_morality>
  <sentiment_author>positive|negative|ambivalent|neutral|not_mentioned</sentiment_author>
</sentiment>

<reviewer_voice>first_person_singular|first_person_plural|impersonal</reviewer_voice>

</review_metadata>

<reviewed_work>
<title></title>
<author>
  <author_name></author_name>
  <author_gender>male|female|unknown|other</author_gender>
</author>
<publisher></publisher>
<category>Gelehrte Literatur|Belletristik|Gebrauchsliteratur</category>

<genre>Roman|Novelle|Erzählung|Drama|Lyrik|Essay|Wissenschaftliche Abhandlung|Ratgeber|Lehrbuch|Schrift|Kalender/Almanach|Lexikon|unknown|other</genre>
<translation>
  <is_translation>true|false|unknown</is_translation>
<source_language>German|French|English|Italian|Latin|Spanish|Russian|unknown|other</source_language>
</translation>

</reviewed_work>

<mentions>

<works>
  <work>
    <title></title>
    <author></author>
  </work>
</works>

<authors>
  <author></author>
</authors>

</mentions>

<quality_control>
  <completeness>complete| fragment </completeness>
  <human_verification_needed>true|false</human_verification_needed>
</quality_control>

```

In [ ]:
!pip install rapidfuzz
import pandas as pd
import re
from rapidfuzz import fuzz

# ------------------------
# Helpers
# ------------------------

def extract_review_block(text):
    if text is None:
        return None
    match = re.search(r"<review>(.*?)</review>", text, re.DOTALL | re.IGNORECASE)
    return match.group(1) if match else text


def remove_mentions_block(text):
    if text is None:
        return None
    return re.sub(
        r"<mentions>.*?</mentions>",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )


def extract_all_tags(text, tag):
    if text is None:
        return []

    pattern = rf"<{tag}>(.*?)</{tag}>"
    matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)

    return [m.strip() for m in matches if m.strip()]


def normalize_text(text):
    if text is None:
        return ""
    text = text.lower()
    text = re.sub(r"[\"'“”‘’.,;:!?()\[\]{}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_human_verification(text):
    vals = extract_all_tags(text, "human_verification_needed")

    if not vals:
        return "unclear"

    val = vals[0].strip().lower()

    if val in {"true", "false"}:
        return val
    return "unclear"


# ------------------------
# Schema
# ------------------------

MANDATORY_FIELDS = {
    "sentiment_overall": "review_metadata/sentiment/sentiment_overall",
    "sentiment_style": "review_metadata/sentiment/sentiment_style",
    "sentiment_content": "review_metadata/sentiment/sentiment_content",
    "sentiment_society": "review_metadata/sentiment/sentiment_society",
    "sentiment_morality": "review_metadata/sentiment/sentiment_morality",
    "sentiment_author": "review_metadata/sentiment/sentiment_author",
    "reviewer_voice": "review_metadata/reviewer_voice",

    "title": "reviewed_work/title",
    "author_name": "reviewed_work/author/author_name",
    "author_gender": "reviewed_work/author/author_gender",
    "publisher": "reviewed_work/publisher",
    "category": "reviewed_work/category",
    "genre": "reviewed_work/genre",
    "is_translation": "reviewed_work/translation/is_translation",
    "source_language": "reviewed_work/translation/source_language",

    "completeness": "quality_control/completeness",
    # removed from scoring:
    # "human_verification_needed"
}


ALLOWED_VALUES = {
    "sentiment_overall": {"positive", "negative", "ambivalent", "neutral"},
    "sentiment_style": {"positive", "negative", "ambivalent", "neutral", "not_mentioned"},
    "sentiment_content": {"positive", "negative", "ambivalent", "neutral", "not_mentioned"},
    "sentiment_society": {"positive", "negative", "ambivalent", "neutral", "not_mentioned"},
    "sentiment_morality": {"positive", "negative", "ambivalent", "neutral", "not_mentioned"},
    "sentiment_author": {"positive", "negative", "ambivalent", "neutral", "not_mentioned"},
    "reviewer_voice": {"first_person_singular", "first_person_plural", "impersonal"},

    "author_gender": {"male", "female", "unknown", "other"},
    "category": {"Gelehrte Literatur", "Belletristik", "Gebrauchsliteratur"},
    "genre": {"Roman","Novelle","Erzählung","Drama","Lyrik","Essay",
              "Wissenschaftliche Abhandlung","Ratgeber","Lehrbuch",
              "Schrift","Kalender/Almanach","Lexikon","unknown","other"},
    "is_translation": {"true", "false", "unknown"},
    "source_language": {"German","French","English","Italian","Latin",
                        "Spanish","Russian","unknown","other"},
    "completeness": {"complete", "fragment"},
}


# ------------------------
# Evaluation
# ------------------------

def evaluate_row(gt_text, llm_text, fuzzy_threshold=80):
    # --- isolate relevant content ---
    gt_text = extract_review_block(gt_text)
    llm_text = extract_review_block(llm_text)

    gt_text = remove_mentions_block(gt_text)
    llm_text = remove_mentions_block(llm_text)

    missing_tags = []
    correct_tags = []
    incorrect_tags = []

    total_fields = len(MANDATORY_FIELDS)

    for field, path in MANDATORY_FIELDS.items():
        tag = path.split("/")[-1]

        gt_vals = extract_all_tags(gt_text, tag)
        llm_vals = extract_all_tags(llm_text, tag)

        # --- Missing ---
        if not llm_vals:
            missing_tags.append(field)
            continue

        llm_val = llm_vals[0]

        # --- Controlled vocab ---
        if field in ALLOWED_VALUES:
            if llm_val not in ALLOWED_VALUES[field]:
                incorrect_tags.append(field)
                continue

        # --- Fuzzy match fields ---
        if field in {"title", "author_name", "publisher"}:
            llm_norm = normalize_text(llm_val)

            match_found = False
            for gt_val in gt_vals:
                score = fuzz.ratio(normalize_text(gt_val), llm_norm)
                if score >= fuzzy_threshold:
                    match_found = True
                    break

            if match_found:
                correct_tags.append(field)
            else:
                incorrect_tags.append(field)

        # --- Exact match fields ---
        else:
            if any(gt_val == llm_val for gt_val in gt_vals):
                correct_tags.append(field)
            else:
                incorrect_tags.append(field)

    # --- human verification (NOT scored) ---
    human_verification = extract_human_verification(llm_text)

    # --- Scores ---
    correct_n = len(correct_tags)
    missing_n = len(missing_tags)
    incorrect_n = len(incorrect_tags)

    return {
        "correct_tags": correct_tags,
        "missing_tags": missing_tags,
        "incorrect_tags": incorrect_tags,
        "prop_correct": round(correct_n / total_fields, 2),
        "prop_missing": round(missing_n / total_fields, 2),
        "prop_incorrect": round(incorrect_n / total_fields, 2),
        "success_score": round(correct_n / total_fields, 2),
        "human_verification_needed": human_verification,
    }


# ------------------------
# Apply to DataFrame
# ------------------------

def run_evaluation(df, llm_column, gt_column="annotation"):
    results = df.apply(
        lambda row: evaluate_row(row[gt_column], row[llm_column]),
        axis=1
    )

    eval_col_name = llm_column + "_evaluation"

    df = df.copy()
    df[eval_col_name] = results

    return df

In [ ]:
from collections import defaultdict
import pandas as pd

def summarize_success(df, evaluation_column):
    """
    Summarizes evaluation results across a dataset.

    - average success score
    - per-tag: correct / missing / incorrect frequencies
    - human_verification_needed counts
    """

    n = len(df)

    tag_counts = defaultdict(lambda: {
        "correct": 0,
        "missing": 0,
        "incorrect": 0
    })

    human_verification_counts = {
        "true": 0,
        "false": 0,
        "unclear": 0
    }

    success_scores = []

    for val in df[evaluation_column]:

        if not isinstance(val, dict):
            continue

        # --- success score ---
        if "success_score" in val:
            success_scores.append(val["success_score"])

        # --- correct / missing / incorrect tags ---
        for tag in val.get("correct_tags", []):
            tag_counts[tag]["correct"] += 1

        for tag in val.get("missing_tags", []):
            tag_counts[tag]["missing"] += 1

        for tag in val.get("incorrect_tags", []):
            tag_counts[tag]["incorrect"] += 1

        # --- human verification ---
        hv = val.get("human_verification_needed", "unclear")
        hv = str(hv).lower()

        if hv in human_verification_counts:
            human_verification_counts[hv] += 1
        else:
            human_verification_counts["unclear"] += 1

    # ------------------------
    # Build summary output
    # ------------------------

    summary_df = pd.DataFrame.from_dict(tag_counts, orient="index")

    summary_df["correct_rate"] = summary_df["correct"] / n
    summary_df["missing_rate"] = summary_df["missing"] / n
    summary_df["incorrect_rate"] = summary_df["incorrect"] / n

    result = {
        "avg_success_score": sum(success_scores) / len(success_scores) if success_scores else 0,
        "dataset_size": n,
        "human_verification_counts": human_verification_counts,
        "tag_summary": summary_df.sort_values("correct_rate", ascending=False)
    }

    return result

In [ ]:

# ------------------------
# Apply to DataFrame
# -----------------------

results = df.apply(
    lambda row: evaluate_row(row["annotation"], row["annotation"]),  # test mode
    axis=1
)

results_df = pd.json_normalize(results)
df_final = pd.concat([df, results_df], axis=1)

In [ ]:
df_final[df_final["success_score"] < 1.0][
    ["annotation", "missing_tags", "incorrect_tags"]
].head(10)

,annotation,missing_tags,incorrect_tags
0,<review>\n\n<review_metadata>\n\n<sentiment>\n...,"[completeness, human_verification_needed]",[]
1,<review>\n\n<review_metadata>\n\n<sentiment>\n...,[],[human_verification_needed]
4,<review>\n\n<review_metadata>\n\n<sentiment>\n...,[],[human_verification_needed]
6,<review>\n\n<review_metadata>\n\n<sentiment>\n...,[],[human_verification_needed]


# Evaluation: Modellvergleich

Es liegt nicht für alle 633 Rezensionen Ground Truth vor - aber es ist möglich, zwischen den beiden Modellen die Übereinstimmung zu überprüfen und damit die Extraktionen/Klassifikationen zu vergleichen.

In [ ]:
df = pd.read_csv("df_reviews_big_with_gpt_and_qwen.csv") # Datenset einlesen

In [ ]:
df_final = run_evaluation(df, llm_column="qwen3_235b_output_big", gt_column="gpt_oss_120b_output_big")

In [ ]:
df_final = run_evaluation(df_final, llm_column="gpt_oss_120b_output_big", gt_column="qwen3_235b_output_big")

In [ ]:
df_final["success_score"] = df_final["annotation_evaluation"].apply(
    lambda x: x.get("success_score") if isinstance(x, dict) else None
)

In [ ]:
summary_gpt = summarize_success(df, "gpt_oss_120b_output_2_evaluation")

In [ ]:
display_df = (
    summary_gpt["tag_summary"]
    .rename(columns={
        "correct": "agreement",
        "incorrect": "disagreement",
        "correct_rate": "agreement_rate",
        "incorrect_rate": "disagreement_rate"
    })
    .drop(columns=["missing", "missing_rate"])
)

display(display_df)

,agreement,disagreement,agreement_rate,disagreement_rate
sentiment_morality,19,1,0.95,0.05
sentiment_society,19,1,0.95,0.05
author_name,18,2,0.90,0.10
reviewer_voice,18,2,0.90,0.10
is_translation,18,2,0.90,0.10
category,18,2,0.90,0.10
source_language,18,2,0.90,0.10
title,17,3,0.85,0.15
completeness,16,4,0.80,0.20
sentiment_author,16,4,0.80,0.20


In [ ]:
print("GPT SUMMARY")
print(summary_gpt["avg_success_score"])
print(summary_gpt["human_verification_counts"])
display(summary_gpt["tag_summary"])

print("\nQWEN SUMMARY")
print(summary_qwen["avg_success_score"])
print(summary_qwen["human_verification_counts"])
display(summary_qwen["tag_summary"])

GPT SUMMARY
0.8175000000000001
{'true': 0, 'false': 20, 'unclear': 0}


,correct,missing,incorrect,correct_rate,missing_rate,incorrect_rate
sentiment_morality,19,0,1,0.95,0.00,0.05
sentiment_society,19,0,1,0.95,0.00,0.05
author_name,18,0,2,0.90,0.00,0.10
reviewer_voice,18,0,2,0.90,0.00,0.10
is_translation,18,0,2,0.90,0.00,0.10
category,18,0,2,0.90,0.00,0.10
source_language,18,0,2,0.90,0.00,0.10
title,17,0,3,0.85,0.00,0.15
completeness,16,0,4,0.80,0.00,0.20
sentiment_author,16,0,4,0.80,0.00,0.20



QWEN SUMMARY


NameError: name 'summary_qwen' is not defined

In [ ]:
from collections import defaultdict
import pandas as pd


def summarize_success_by_hv_any(df, col_a, col_b):
    """
    Groups:
    - hv_any_true  (at least one model requests human verification)
    - hv_none_true (neither model requests it)
    """

    def get_hv(val):
        if not isinstance(val, dict):
            return "unclear"
        hv = val.get("human_verification_needed", "unclear")
        return str(hv).lower()

    groups = {
        "hv_any_true": [],
        "hv_none_true": []
    }

    # ------------------------
    # Assign rows to groups
    # ------------------------
    for _, row in df.iterrows():

        val_a = row[col_a]
        val_b = row[col_b]

        hv_a = get_hv(val_a)
        hv_b = get_hv(val_b)

        if hv_a == "true" or hv_b == "true":
            group = "hv_any_true"
        else:
            group = "hv_none_true"

        groups[group].append((val_a, val_b))

    # ------------------------
    # Summarize per group
    # ------------------------
    results = {}

    for group_name, rows in groups.items():

        tag_counts = defaultdict(lambda: {
            "correct": 0,
            "missing": 0,
            "incorrect": 0
        })

        success_scores = []
        n = len(rows)

        for val_a, val_b in rows:

            val = val_a  # still using model A as reference

            if not isinstance(val, dict):
                continue

            if "success_score" in val:
                success_scores.append(val["success_score"])

            for tag in val.get("correct_tags", []):
                tag_counts[tag]["correct"] += 1

            for tag in val.get("missing_tags", []):
                tag_counts[tag]["missing"] += 1

            for tag in val.get("incorrect_tags", []):
                tag_counts[tag]["incorrect"] += 1

        summary_df = pd.DataFrame.from_dict(tag_counts, orient="index")

        if n > 0:
            summary_df["correct_rate"] = summary_df["correct"] / n
            summary_df["missing_rate"] = summary_df["missing"] / n
            summary_df["incorrect_rate"] = summary_df["incorrect"] / n

        results[group_name] = {
            "n": n,
            "avg_success_score": sum(success_scores) / len(success_scores) if success_scores else 0,
            "tag_summary": summary_df.sort_values("correct_rate", ascending=False)
                if not summary_df.empty else summary_df
        }

    return results

In [ ]:
summary_groups = summarize_success_by_hv_any(
    df_final,
    "qwen3_235b_output_big_evaluation",
    "gpt_oss_120b_output_big_evaluation"
)

In [ ]:
group = "hv_any_true"

display_df = (
    summary_groups[group]["tag_summary"]
    .rename(columns={
        "correct": "agreement",
        "incorrect": "disagreement",
        "correct_rate": "agreement_rate",
        "incorrect_rate": "disagreement_rate"
    })
    .drop(columns=["missing", "missing_rate"])
    .sort_values("agreement_rate", ascending=False)
)

agreement_score = summary_groups[group]["avg_success_score"]

print("Agreement GPT OSS 120B vs. Qwen3 235B VL - model(s) requesting human verification")

print("---")

print("Agreement proportion:", agreement_score)

display(display_df)

Agreement GPT OSS 120B vs. Qwen3 235B VL - model(s) requesting human verification
---
Agreement proportion: 0.7128648648648649


,agreement,disagreement,agreement_rate,disagreement_rate
sentiment_morality,168,17,0.908108,0.091892
is_translation,157,28,0.848649,0.151351
source_language,149,36,0.805405,0.194595
author_name,141,44,0.762162,0.237838
title,139,46,0.751351,0.248649
category,135,50,0.729730,0.270270
author_gender,135,50,0.729730,0.270270
sentiment_society,135,50,0.729730,0.270270
sentiment_style,128,57,0.691892,0.308108
sentiment_overall,126,59,0.681081,0.318919


In [ ]:
def add_hv_combined_column(df, qwen_col, gpt_col):

    def get_hv(val):
        if not isinstance(val, dict):
            return "unclear"
        return str(val.get("human_verification_needed", "unclear")).lower()

    def combine(row):

        hv_qwen = get_hv(row[qwen_col])
        hv_gpt = get_hv(row[gpt_col])

        qwen_true = hv_qwen == "true"
        gpt_true = hv_gpt == "true"

        if qwen_true and gpt_true:
            return "true (both models)"
        elif qwen_true:
            return "true (Qwen)"
        elif gpt_true:
            return "true (GPT)"
        else:
            return "false"

    df = df.copy()
    df["human_verification_needed"] = df.apply(combine, axis=1)

    return df

In [ ]:
df_final = add_hv_combined_column(
    df_final,
    "qwen3_235b_output_big_evaluation",
    "gpt_oss_120b_output_big_evaluation"
)